# Voice AI Stack — Colab Test Notebook

Test STT, TTS, and an LLM (via vLLM) individually before wiring them into the full LiveKit pipeline.

**Runtime:** Runtime > Change runtime type > GPU (T4 is fine for this notebook; use A100 if available for the LLM section).

Order: run Section 0, then whichever of Sections 1/2/3 you want to test. They are independent — no need to run all three in one session (loading STT + TTS + an LLM together may exceed T4's 16GB VRAM).

## 0. Environment check

In [ ]:
!nvidia-smi
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

## 1. STT — Speech to Text

Two tracks:
- **1a. faster-whisper** — quickest to get running, good sanity-check baseline.
- **1b. Nemotron 3.5 ASR 0.6B (NeMo)** — the actual production candidate. Heavier install, closer to what you'll deploy.

Run 1a first to confirm your pipeline (audio in → text out) works end-to-end, then move to 1b for the real comparison.

In [ ]:
# 1a. faster-whisper baseline
!pip install -q faster-whisper

# grab a short sample clip to transcribe (swap this for your own uploaded audio)
!wget -q -O sample.wav https://raw.githubusercontent.com/SYSTRAN/faster-whisper/master/tests/data/jfk.flac

from faster_whisper import WhisperModel
model = WhisperModel("large-v3", device="cuda", compute_type="float16")

segments, info = model.transcribe("sample.wav", beam_size=5)
print(f"Detected language: {info.language} (p={info.language_probability:.2f})")
for seg in segments:
    print(f"[{seg.start:.2f}s -> {seg.end:.2f}s] {seg.text}")

In [ ]:
# 1b. Nemotron 3.5 ASR 0.6B via NVIDIA NeMo
# Heavier install — expect a few minutes. Restart runtime if imports fail after install.
!pip install -q "nemo_toolkit[asr]"

import nemo.collections.asr as nemo_asr

asr_model = nemo_asr.models.ASRModel.from_pretrained(
    model_name="nvidia/nemotron-3.5-asr-streaming-0.6b"
)

# Upload your own Indian-accent-English sample for a realistic test:
# from google.colab import files
# uploaded = files.upload()
# audio_path = list(uploaded.keys())[0]

audio_path = "sample.wav"  # replace with your uploaded file
transcription = asr_model.transcribe([audio_path])
print(transcription)

## 2. TTS — Text to Speech

Kokoro-82M first (fast, tiny). Compare accent naturalness against CosyVoice 2 once Kokoro is confirmed working.

In [ ]:
# 2a. Kokoro-82M
!pip install -q kokoro soundfile
!apt-get -qq install espeak-ng > /dev/null

from kokoro import KPipeline
import soundfile as sf
from IPython.display import Audio, display

pipeline = KPipeline(lang_code="a")  # 'a' = American English voice pack; swap per available Indian-English voice
text = "Hi, thanks for joining the interview today. Could you start by telling me about your current role?"

for i, (gs, ps, audio) in enumerate(pipeline(text, voice="af_heart")):
    sf.write(f"kokoro_out_{i}.wav", audio, 24000)
    display(Audio(f"kokoro_out_{i}.wav"))

## 3. LLM — Qwen via vLLM

Start with **Qwen3-4B-Instruct** — fits comfortably on a T4/A100 in Colab. Once the pipeline works, swap `MODEL_NAME` for the 8B/14B/30B-A3B variants on your actual production GPU.

Note: vLLM install/import can conflict with other packages loaded in this session — if you hit errors, it's safest to run this section in a fresh Colab runtime by itself.

In [ ]:
!pip install -q vllm

from vllm import LLM, SamplingParams

MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"  # swap to Qwen3-8B / Qwen3.5-4B / Qwen3-30B-A3B as GPU allows

llm = LLM(model=MODEL_NAME, gpu_memory_utilization=0.85, max_model_len=8192)

sampling_params = SamplingParams(temperature=0.7, top_p=0.9, max_tokens=200)

prompt = (
    "You are an AI interviewer. Ask one thoughtful follow-up question based on this candidate answer: \n"
    "'In my last role I led a team migrating our monolith to microservices, mainly focused on the payments service.'"
)

outputs = llm.generate([prompt], sampling_params)
for out in outputs:
    print(out.outputs[0].text.strip())

In [ ]:
# Optional: run as an OpenAI-compatible server instead of in-process (closer to production)
# This is how you'd actually serve it behind LiveKit / your gateway.
# Run in a terminal / background cell, not inline:
#
# vllm serve Qwen/Qwen3-4B-Instruct-2507 \
#   --port 8000 \
#   --tensor-parallel-size 1 \
#   --max-model-len 8192 \
#   --gpu-memory-utilization 0.85
#
# Then call it like any OpenAI-compatible endpoint:
# from openai import OpenAI
# client = OpenAI(base_url="http://localhost:8000/v1", api_key="not-needed")
# resp = client.chat.completions.create(model="Qwen/Qwen3-4B-Instruct-2507", messages=[{"role":"user","content":"Hello"}])
# print(resp.choices[0].message.content)

## 4. Path from this notebook to production

| Colab step | Production equivalent |
|---|---|
| `LLM(...)` in-process | `vllm serve ...` as a long-running OpenAI-compatible container, behind your AI gateway, with `--tensor-parallel-size` set for your GPU count |
| `asr_model.transcribe([path])` | Nemotron served via NeMo's streaming inference server or Riva, exposed over WebSocket to LiveKit |
| `pipeline(text, voice=...)` | Kokoro/CosyVoice served behind FastAPI (or Triton), streaming audio chunks back to LiveKit as they're generated |
| Single Colab GPU | Separate GPU pools per product (Aaptor vs Racko-style CS), each behind its own autoscaling group — see architecture notes from earlier in this conversation |
| Manual runs | Wrapped in Docker, deployed via Kubernetes (or Ray Serve for the model-serving layer specifically), with OpenTelemetry + Grafana for latency/throughput monitoring, matching the observability layer in your Racko architecture doc |

Suggested test order once each model works individually: wire STT → LLM → TTS together in one script (no LiveKit yet) to measure round-trip latency, *then* integrate into LiveKit's `VoicePipelineAgent`.